**Author:** Brendan OConnell 

**Date:** May 2026  

**Purpose:** XGBoost Feature Engineering using the minimal pre-processed NFI dataset

**UPDATE** 

This feature engineering NB was retroactively created as the main feature pipeline for the XGBoost model after several key findings took place between model exploration and model evaluation:

- The `core_gsr_count` feature appears to be a label proxy

- The `log_pb_plus_sb` feature performed abysmally when evaluated against the ambiguous particle subset. It incorrectly learned to identify high levels of antimony (Sb) as a predictor for GSR, seemingly disregarding whether the other core GSR elements, lead (Pb) amd barium (Ba), had any significant presence. This last statement about the disregard for Pb/Ba weights is a safe assumption due to the known characterization of the ambiguous particle set (it is high in only 1 of the core GSR elements).

Some of the feature exploration steps are also worth reevaluating now that all 89 raw elements are in the pre-processed dataset.

### Original Key Findings from EDA

- Elemental ratios may yield discriminative signal that raw concentrations do not
- Barium = problematic for false positives --> GSR/Non-GSR boundary overlapping (BaCaSi, BaAl, BaSb, PbBa)
- Non-Barium confounders = Ca, Si, Al
- CuZn highly correlated Non-GSR

> `core_gsr_count` was removed from the key EDA findings, as was the mention of oxygen potentially diluting particle composition. After additional exploration in the sandbox with `NFI_oxygen_analysis.ipynb`, the decision was made t

In [66]:
import pandas as pd
import numpy as np

# Using "average precision score" to estimate PR-AUC for standalone feature precision
from sklearn.metrics import average_precision_score as ap_score

## Load minimally preprocessed NFI dataset

In [67]:
df = pd.read_parquet("../../../data/processed/preprocessed_minimal.parquet")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns}")

new_feats = pd.DataFrame()

Shape: (2294985, 94)
Columns: Index(['stub_id', 'particle_id', 'ac', 'ag', 'al', 'ar', 'as', 'at', 'au', 'b',
       'ba', 'bi', 'br', 'ca', 'cd', 'ce', 'cl', 'co', 'cr', 'cs', 'cu', 'dy',
       'er', 'eu', 'f', 'fe', 'fr', 'ga', 'gd', 'ge', 'hf', 'hg', 'ho', 'i',
       'in', 'ir', 'k', 'kr', 'la', 'lu', 'mg', 'mn', 'mo', 'n', 'na', 'nb',
       'nd', 'ne', 'ni', 'np', 'o', 'os', 'p', 'pa', 'pb', 'pd', 'pm', 'po',
       'pr', 'pt', 'pu', 'ra', 'rb', 're', 'rh', 'rn', 'ru', 's', 'sb', 'sc',
       'se', 'si', 'sm', 'sn', 'sr', 'ta', 'tb', 'tc', 'te', 'th', 'ti', 'tl',
       'tm', 'u', 'v', 'w', 'xe', 'y', 'yb', 'zn', 'zr', 'class', 'label',
       'target'],
      dtype='str')


### Custom Functions

Estimating PR-AUC with average_precision_score

In [68]:
def compute_prauc(vals):
    y = (df["target"] == 1).astype(int).values  # GSR
    return round(ap_score(y, vals), 4)

Safe division with a sentinel value (using **-1**)

In [69]:
def sentinel_divide(n, d):
    """
    Divide n by d, but return a sentinel value if 'd=0'.

    Used for feature engineering to avoid 'NaN' and 'inf' quotient when 'd=0'.
    The usage of this is discussed in detail here:
        `notebooks/02_feature_processing/engineering/feature_exploration_xgboost.ipynb`

    Args:
        n: Numerator
        d: Denominator

    Returns:
        Quotient of n/d, or a sentinel value if d=0.

    Original Author: Brendan OConnell
    """
    sentinel = -1
    safe_d = np.where(d == 0, sentinel, d)
    return n / safe_d

**89 raw element features** 

In [70]:
meta_cols = ["stub_id", "particle_id", "class", "label", "target"]
element_cols = [c for c in df.columns if c not in meta_cols]
len(element_cols)

89

-----

Feature that quantifies Pb and Sb.

From the original feature exploring NB, the following scenarios were deemed as reasons for this feature:
- Large quantities of both are weighted higher
- Trace quantities of both are penalized strongly
- A large quantity of 1 and trace quantity of 1 get slightly penalized

The reason for penalizing the 3rd scenario, instead of additive (Pb + Sb) is because a trace amount of any of these elements should be taken with caution as it may be a sign of contamination and false-positives.

## Feature: Pb x Sb (multiplicative quantification)

Based on the original feature exploration, this "Pb x Sb" feature will be complimentary by quantifying Pb and Sb, and penalizing trace amounts to mitigate false-positives in some of the brass and barium-heavy Non-GSR particles.

In [71]:
new_feats["pb_times_sb"] = df["pb"] * (df["sb"])

# Estimate PR-AUC
print(
    f"PR-AUC estimate for pb_times_sb: {compute_prauc(new_feats['pb_times_sb'].values)}"
)

PR-AUC estimate for pb_times_sb: 0.8323


---

### Measuring Percentage of Total Mass:

#### An alternative approach to Element Ratios

An alternative to ratios between direct elements, which has the potential downside of a 0 denominator, we can check the ratio between GSR elements over the total mass of the particle.

- (Pb+Ba) / (total mass - Sb)
- (Pb+Sb) / (total mass - Ba)
- (Ba+Sb) / (total mass - Pb)
- (Cu+Zn) / (total mass)
- (Ti+Zn) / (total mass)

The inclusion of Cu&Zn is for the Non-GSR brass particles that have core GSR elements present.

The inclusion of Ti&Zn is for the Non-GSR TiZnGd & TiZn Non-GSR glasses with core GSR count of 2.

__Calculate Mass (denominator)__

In [72]:
total_mass = df[element_cols].sum(axis=1)

# PbBa numerator
total_mass_no_sb = df[element_cols].sum(axis=1) - df["sb"]

# PbSb numerator
total_mass_no_ba = df[element_cols].sum(axis=1) - df["ba"]

# BaSb numerator
total_mass_no_pb = df[element_cols].sum(axis=1) - df["pb"]

__Pb+Ba / (mass - Sb)

In [73]:
new_feats["pb_ba_over_non_sb_mass"] = (df["pb"] + df["ba"]) / total_mass_no_sb

# estimate pr-auc
print(
    f"PR-AUC estimate for pb_ba_over_non_sb_mass: {compute_prauc(new_feats['pb_ba_over_non_sb_mass'].values)}"
)

PR-AUC estimate for pb_ba_over_non_sb_mass: 0.6913


__Pb+Sb / (mass - Ba)__

In [74]:
new_feats["pb_sb_over_non_ba_mass"] = (df["pb"] + df["sb"]) / total_mass_no_ba


# estimate pr-auc
print(
    f"PR-AUC estimate for pb_sb_over_non_ba_mass: {compute_prauc(new_feats['pb_sb_over_non_ba_mass'].values)}"
)

PR-AUC estimate for pb_sb_over_non_ba_mass: 0.9979


__Ba+Sb / (mass - Pb)__

In [75]:
new_feats["ba_sb_over_non_pb_mass"] = (df["ba"] + df["sb"]) / total_mass_no_pb

# estimate pr-auc
print(
    f"PR-AUC estimate for ba_sb_over_non_pb_mass: {compute_prauc(new_feats['ba_sb_over_non_pb_mass'].values)}"
)

PR-AUC estimate for ba_sb_over_non_pb_mass: 0.6158


__Cu+Zn / (mass)__

In [76]:
new_feats["cu_zn_over_mass"] = (df["cu"] + df["zn"]) / total_mass

# estimate pr-auc
print(
    f"PR-AUC estimate for cu_zn_over_mass: {compute_prauc(new_feats['cu_zn_over_mass'].values)}"
)

PR-AUC estimate for cu_zn_over_mass: 0.4336


__Ti+Zn / (mass)__

In [77]:
new_feats["ti_zn_over_mass"] = (df["ti"] + df["zn"]) / total_mass

# estimate pr-auc
print(
    f"PR-AUC estimate for ti_zn_over_mass: {compute_prauc(new_feats['ti_zn_over_mass'].values)}"
)

PR-AUC estimate for ti_zn_over_mass: 0.4012


_____
## Feature: Environmental & Mineral Confounders

### (domain knowledge)

Based on a combination of EDA, feature exploration, and domain knowledge, the top GSR confounders include:
- Ca
- Si
- Al
- Fe

These are often found in things such as fireworks, brakepads, industrial materials, and environmental particles.

This feature will calculate the ratio of GSR element mass over the confounder element mass, but specifically removing barium since it is heavily present in both GSR and Non-GSR particles.

> **Note:**
Based on the original **`feature_exploration_xgboost.ipynb`** (moved to 99_sandbox to replace with this verion), it was determined to use a custom function for `sentinel divide` instead of `divide_with_eps`. See below.

An alternative to *safe-divide-with-epsilon* is to use a sentinel value that signals `denominator=0` to be deemed undefined. For example, when __d=0__ replace it with __-1__ so that any `inf` scenario outputs a negative quotient. The hope would be that a tree model like xgboost would identify the threshold between positive and negative numbers, interpreting positive quotients as reliable signals and negative quotients as unreliable and instead would rely on other features. This also address the `NaN` values, since `0 / (-1)` is 0, which is an appropriate value since all 191 NaN rows are Non-GSR.

In [78]:
gsr = df["pb"] + df["sb"]
confounders = df["ca"] + df["si"] + df["al"] + df["fe"]
new_feats["gsr_over_confounders"] = sentinel_divide(gsr, confounders)

# estimate pr-auc
print(
    f"PR-AUC estimate for gsr_over_confounders: {compute_prauc(new_feats['gsr_over_confounders'].values)}"
)

PR-AUC estimate for gsr_over_confounders: 0.7897


### Check for any **'inf'** or **'nan'** values to confirm that the sentinel-safe-divide function is working.

In [79]:
any(
    np.isinf(new_feats["gsr_over_confounders"])
    | new_feats["gsr_over_confounders"].isna()
)

False

Check to ensure no whacky max or min vals like with epsilon

In [80]:
print(
    f"Max val: {new_feats['gsr_over_confounders'].max()}\nMin val: {new_feats['gsr_over_confounders'].min()}"
)

Max val: 291.46565724628874
Min val: -100.00000381469727


In [81]:
# estimate pr-auc
print(
    f"PR-AUC estimate for gsr_over_confounders: {compute_prauc(new_feats['gsr_over_confounders'].values)}"
)

PR-AUC estimate for gsr_over_confounders: 0.7897


-----
# Final Engineered Features to Consider

In [82]:
final_feats = pd.DataFrame(
    {
        "NewFeature": new_feats.columns,
        "PR-AUC estimate": [
            compute_prauc(new_feats[col].values) for col in new_feats.columns
        ],
    }
)

final_feats.sort_values("PR-AUC estimate", ascending=False).reset_index(
    drop=True
).rename(lambda x: x + 1).style.set_properties(
    subset=["NewFeature"], **{"text-align": "left"}
)

,NewFeature,PR-AUC estimate
1,pb_sb_over_non_ba_mass,0.997900
2,pb_times_sb,0.832300
3,gsr_over_confounders,0.789700
4,pb_ba_over_non_sb_mass,0.691300
5,ba_sb_over_non_pb_mass,0.615800
6,cu_zn_over_mass,0.433600
7,ti_zn_over_mass,0.401200


In [83]:
# create the engineered dataset with preprocessed columns + engineered features
engineered_df = df[
    ["stub_id", "particle_id", "label", "target", "class"] + element_cols
].copy()
engineered_df = pd.concat([engineered_df, new_feats], axis=1)

print(f"Engineered Dataset shape: {engineered_df.shape}")
print(f"\nEngineered Dataset columns: {engineered_df.columns}")

Engineered Dataset shape: (2294985, 101)

Engineered Dataset columns: Index(['stub_id', 'particle_id', 'label', 'target', 'class', 'ac', 'ag', 'al',
       'ar', 'as',
       ...
       'yb', 'zn', 'zr', 'pb_times_sb', 'pb_ba_over_non_sb_mass',
       'pb_sb_over_non_ba_mass', 'ba_sb_over_non_pb_mass', 'cu_zn_over_mass',
       'ti_zn_over_mass', 'gsr_over_confounders'],
      dtype='str', length=101)


**Instead of writing to parquet, this will be stored as a Python script and invoked for the Model Baseline Exploration section (due to LFS data constraints on my account).**

In [84]:
# output the engineered dataset to be used with xgboost model
# engineered_df.to_parquet("../../../data/processed/engineered_features_xgboost.parquet")

# NFI engineered data:

In [85]:
print(f"Raw Element features = {len(element_cols)}")
print(f"Engineered features = {len(new_feats.columns)}")
print(
    f"Total features (raw + engineered) = {len(element_cols) + len(new_feats.columns)}"
)

Raw Element features = 89
Engineered features = 7
Total features (raw + engineered) = 96


---
---
---
---
---

**A note on potential Bias (avoided by not using epsilon approach)**

*From the original feature exploration, an explanation of the decision to not use an epsilon formula for safe dividing ratios that have 0 in denominator, and instead proposing the usage of a sentinel value like **-1***

#### Potential bias with epsilon approach for handling 'inf' ... didn't account for how much it would inflate the ratio.... need to review if this approach is appropriate.

Another thing to consider here is that in the full dataset (which includes ambiguous labels) if Pb>0 and Ba=0,  this scenario could be where Pb is the only core GSR element present, in which case this ratio may not be a value that the model should use to classify the label.

Therefore, an alternative approach might be to use a sentinel value that signals Ba=0 (or any denominator as 0) to be deemed unreliable. For example, when __d=0__ replace it with __-1__ so that any `inf` scenario outputs a negative quotient. The hope would be that a tree model like xgboost would identify the threshold between positive and negative numbers, interpreting positive quotients as reliable signals and negative quotients as unreliable and instead would rely on other features.